# Vietnamese Legal RAG — Grounded Synthetic QA Generation (G-LRAG v2)

Notebook nay sinh QA benchmark theo pipeline trong `vietnamese_legal_rag_benchmark_final_v3.md`:

`Representative Sampling -> Provision Role Classification -> Grounded QA Generation -> Minimal Evidence Tagging -> LLM Verification -> checkpoint -> export`

**Truoc khi chay:**
1. Add corpus (`documents.jsonl`, `provisions.jsonl`, `chunks.jsonl`, `edges.jsonl`, `validity_timeline.jsonl`, `text_provenance.jsonl`) nhu 1 Kaggle Dataset, mount vao `/kaggle/input/glrag-v2/`.
2. Add Anthropic API key qua **Add-ons > Secrets** (xem huong dan rieng: `kaggle_api_key_guide.md`).
3. Bat **Internet: ON** trong Notebook Settings (bat buoc de goi Anthropic API).


## 1. Cai dat thu vien

In [1]:
# !pip install -q anthropic tqdm


In [2]:
!pip install -q google-genai tqdm

## 2. Lay API key tu Kaggle Secrets (KHONG hardcode key vao notebook)

In [18]:
from kaggle_secrets import UserSecretsClient
from google import genai
from google.genai import types

user_secrets = UserSecretsClient()

client = genai.Client(
    api_key=user_secrets.get_secret("GEMINI_API_KEY")
)

## 3. Config

In [ ]:
import json, time, random, hashlib, math
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from tqdm.notebook import tqdm

CONFIG = {
    "v2_data_root": "/kaggle/input/datasets/phuongthao205/legalrag/pre-processed",
    "output_dir": "/kaggle/working/qa_output",
    "checkpoint_path": "/kaggle/working/qa_output/checkpoint.jsonl",

    # Intermediate artifacts written from the same run/checkpoint.
    "qa_candidates_path": "/kaggle/working/qa_output/qa_candidates.jsonl",
    "qa_verified_path": "/kaggle/working/qa_output/qa_verified.jsonl",
    "qa_final_path": "/kaggle/working/qa_output/qa_final.jsonl",
    "run_report_path": "/kaggle/working/qa_output/qa_generation_report.json",
    "run_report_md_path": "/kaggle/working/qa_output/qa_generation_report.md",

    # Gemini 3.1 Flash Lite for low-cost bulk generation/verification.
    "generator_model": "gemini-3.1-flash-lite",
    "verifier_model": "gemini-3.1-flash-lite",

    # Bundle prompts ask for 5-6 QA across categories, so allow a little more output.
    "max_tokens_gen": 2600,
    "max_tokens_verify": 1600,

    # Final benchmark target: 400 general QA + 100 Vietnamese legal nuance QA.
    "main_target_total": 400,
    "nuance_target_total": 100,
    "target_total": 500,
    "quota": {
        "single_hop": 140,
        "citation": 60,
        "multi_hop": 60,
        "cross_document": 50,
        "legal_validity": 60,
        "unanswerable": 30,
    },

    # Diversity is now primarily generated up front: one task = one related context bundle.
    # Each bundle contains 3-5 provisions/chunks and asks for 5-6 QA across categories.
    "oversample_factor": 1.8,
    "batch_size": 6,
    "context_units_per_task": 5,
    "max_qa_per_center_in_request": 2,
    "sampled_provisions_target": 1800,
    "max_bundles_per_seed_document": 1,
    "nuance_oversample_factor": 1.2,
    "nuance_verify_batch_size": 6,
    "nuance_quota": {
        "continuous_amendment": 20,
        "partial_validity": 20,
        "multi_tier_relation": 20,
        "conflict_resolution": 20,
        "buffer_single_citation": 20,
    },
    "min_provisions_per_sampled_doc": 2,
    "max_provisions_per_sampled_doc": 5,
    "min_chunk_chars_for_seed": 80,

    # Gemini text-out limits: RPM=15, TPM=250K, RPD=500.
    # Use 80% RPM => effective 12 RPM, equivalent to at least ~5s/request.
    "requests_per_minute": 15,
    "tokens_per_minute": 250_000,
    "requests_per_day": 500,
    "rpm_safety_factor": 0.80,
    "min_request_interval_sec": 5.0,
    "max_workers": 2,

    # Expected: ~240 general calls + ~145 nuance calls before retry, below RPD=500.
    "max_api_requests": 480,

    # Safety net only; generation already limits per-provision concentration in each bundle.
    "max_qa_per_provision": 2,
    "max_qa_per_document_ratio": 0.03,
    "dedup_exact_questions": True,

    "corpus_version": "glrag-v2-2026-07-snapshot",
    "as_of_date": "2026-07-13",

    "retry_max": 3,
    "retry_backoff_sec": 20,
}

Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

## 4. Load corpus G-LRAG v2

In [ ]:
def iter_jsonl(root, name):
    path = Path(root) / name
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def peek_jsonl(root, name, n=1):
    rows = []
    for row in iter_jsonl(root, name):
        rows.append(row)
        if len(rows) >= n:
            break
    return rows

def _doc_field_code(doc):
    legal_field = doc.get("legal_field") or {}
    if isinstance(legal_field, dict):
        return legal_field.get("code", "UNMAPPED")
    return str(legal_field or "UNMAPPED")

def load_corpus(root):
    # Stream each JSONL once and keep only the compact indexes used by the QA notebook.
    # Dataset v2 keys:
    # - documents.id_str is the document PK
    # - provisions.unit_id is the provision PK, provisions.id_str is the document FK
    # - chunks.parent_unit_id is the provision FK; chunks are hydrated lazily after sampling
    # - validity_timeline/text_provenance are document-level, keyed by id_str
    documents_by_id = {}
    for d in iter_jsonl(root, "documents.jsonl"):
        doc_id = d.get("id_str")
        if not doc_id:
            continue
        documents_by_id[doc_id] = {
            "id_str": doc_id,
            "title": d.get("title") or d.get("ten_van_ban") or d.get("name") or doc_id,
            "legal_field": d.get("legal_field"),
            "currency_hint": d.get("currency_hint"),
            "legal_authority_rank": d.get("legal_authority_rank"),
        }

    provisions_by_id = {}
    provision_ids_by_doc = defaultdict(list)
    for p in iter_jsonl(root, "provisions.jsonl"):
        unit_id = p.get("unit_id")
        doc_id = p.get("id_str")
        if not unit_id or not doc_id:
            continue
        provisions_by_id[unit_id] = {
            "unit_id": unit_id,
            "id_str": doc_id,
            "citation_anchor": p.get("citation_anchor", ""),
            "path": p.get("path", ""),
            "coverage_verified": p.get("coverage_verified"),
        }
        provision_ids_by_doc[doc_id].append(unit_id)

    edges_by_src_doc = defaultdict(list)
    edges_by_dst_doc = defaultdict(list)
    verified_edge_count = 0
    for e in iter_jsonl(root, "edges.jsonl"):
        if e.get("direction_verified") is not True:
            continue
        src_id = e.get("src_id")
        if not src_id:
            continue
        edge = {
            "src_id": src_id,
            "dst_id": e.get("dst_id"),
            "rel_canonical": e.get("rel_canonical"),
            "external_target": e.get("external_target"),
            "direction_verified": True,
        }
        edges_by_src_doc[src_id].append(edge)
        if edge.get("dst_id"):
            edges_by_dst_doc[edge["dst_id"]].append(edge)
        verified_edge_count += 1

    validity_by_doc = defaultdict(list)
    for v in iter_jsonl(root, "validity_timeline.jsonl"):
        doc_id = v.get("id_str")
        if doc_id:
            validity_by_doc[doc_id].append(v)

    provenance_by_doc = {}
    for tp in iter_jsonl(root, "text_provenance.jsonl"):
        doc_id = tp.get("id_str")
        if doc_id:
            provenance_by_doc[doc_id] = tp

    return {
        "data_root": str(root),
        "documents_by_id": documents_by_id,
        "provisions_by_id": provisions_by_id,
        "provision_ids_by_doc": provision_ids_by_doc,
        "chunks_by_provision": defaultdict(list),
        "edges_by_src_doc": edges_by_src_doc,
        "edges_by_dst_doc": edges_by_dst_doc,
        "verified_edge_count": verified_edge_count,
        "validity_by_doc": validity_by_doc,
        "provenance_by_doc": provenance_by_doc,
    }

def hydrate_chunks_for_units(corpus, unit_ids, root=None):
    root = root or corpus["data_root"]
    target_unit_ids = {uid for uid in unit_ids if uid and uid in corpus["provisions_by_id"]}
    already_loaded = {uid for uid in target_unit_ids if uid in corpus["chunks_by_provision"]}
    missing_unit_ids = target_unit_ids - already_loaded
    if not missing_unit_ids:
        return 0

    loaded = 0
    for c in iter_jsonl(root, "chunks.jsonl"):
        parent_unit_id = c.get("parent_unit_id")
        if parent_unit_id not in missing_unit_ids:
            continue
        corpus["chunks_by_provision"][parent_unit_id].append({
            "chunk_id": c.get("chunk_id"),
            "parent_unit_id": parent_unit_id,
            "id_str": c.get("id_str"),
            "chunk_index_in_unit": c.get("chunk_index_in_unit", 0),
            "chunk_text": c.get("chunk_text", ""),
        })
        loaded += 1

    for chunks in corpus["chunks_by_provision"].values():
        chunks.sort(key=lambda c: c.get("chunk_index_in_unit", 0))
    return loaded

corpus = load_corpus(CONFIG["v2_data_root"])
print(f"Documents: {len(corpus['documents_by_id'])}")
print(f"Provisions: {len(corpus['provisions_by_id'])}")
print(f"Verified edges: {corpus['verified_edge_count']}")
print(f"Validity-tagged documents: {len(corpus['validity_by_doc'])}")

## 5. Seed Sampling For Related Context Bundles

In [ ]:
def document_score_for_sampling(corpus, doc_id):
    score = 0
    if corpus["provenance_by_doc"].get(doc_id, {}).get("text_status") == "available":
        score += 3
    if corpus["edges_by_src_doc"].get(doc_id) or corpus.get("edges_by_dst_doc", {}).get(doc_id):
        score += 2
    if corpus["validity_by_doc"].get(doc_id):
        score += 2
    if corpus["documents_by_id"].get(doc_id, {}).get("legal_authority_rank") is not None:
        score += 1
    return score


def representative_units_for_doc(corpus, doc_id, n_units):
    unit_ids = corpus["provision_ids_by_doc"].get(doc_id, [])
    if not unit_ids:
        return []
    if len(unit_ids) <= n_units:
        return list(unit_ids)
    if n_units == 1:
        return [unit_ids[len(unit_ids) // 2]]
    positions = [round(i * (len(unit_ids) - 1) / (n_units - 1)) for i in range(n_units)]
    selected = []
    for pos in positions:
        uid = unit_ids[pos]
        if uid not in selected:
            selected.append(uid)
    return selected


def sample_documents_then_provisions(corpus, target_provisions, min_per_doc=2, max_per_doc=5,
                                     cap_ratio_per_field=0.25, seed=42):
    rng = random.Random(seed)
    docs_by_field = defaultdict(list)
    for doc_id, unit_ids in corpus["provision_ids_by_doc"].items():
        if len(unit_ids) < min_per_doc:
            continue
        doc = corpus["documents_by_id"].get(doc_id, {})
        field = _doc_field_code(doc)
        docs_by_field[field].append(doc_id)

    avg_units_per_doc = max(min_per_doc, min(max_per_doc, (min_per_doc + max_per_doc) / 2))
    target_docs = max(1, math.ceil(target_provisions / avg_units_per_doc))
    per_field_cap = max(1, int(target_docs * cap_ratio_per_field))
    sampled_docs = []
    for field, doc_ids in docs_by_field.items():
        doc_ids = list(doc_ids)
        rng.shuffle(doc_ids)
        doc_ids.sort(key=lambda d: document_score_for_sampling(corpus, d), reverse=True)
        sampled_docs.extend(doc_ids[:per_field_cap])

    # Fill remaining docs globally by metadata richness while keeping broad field coverage first.
    if len(sampled_docs) < target_docs:
        selected = set(sampled_docs)
        remaining = [doc_id for ids in docs_by_field.values() for doc_id in ids if doc_id not in selected]
        rng.shuffle(remaining)
        remaining.sort(key=lambda d: document_score_for_sampling(corpus, d), reverse=True)
        sampled_docs.extend(remaining[: target_docs - len(sampled_docs)])

    rng.shuffle(sampled_docs)
    sampled_docs = sampled_docs[:target_docs]

    sampled_unit_ids = []
    for i, doc_id in enumerate(sampled_docs):
        available = len(corpus["provision_ids_by_doc"].get(doc_id, []))
        span = max(1, max_per_doc - min_per_doc + 1)
        requested = min_per_doc + (i % span)
        n_units = min(max_per_doc, max(min_per_doc, min(available, requested)))
        sampled_unit_ids.extend(representative_units_for_doc(corpus, doc_id, n_units))
        if len(sampled_unit_ids) >= target_provisions:
            break

    sampled_unit_ids = list(dict.fromkeys(sampled_unit_ids))[:target_provisions]
    sampled_doc_ids = list(dict.fromkeys(corpus["provisions_by_id"].get(uid, {}).get("id_str")
                                         for uid in sampled_unit_ids
                                         if corpus["provisions_by_id"].get(uid, {}).get("id_str")))
    return sampled_doc_ids, sampled_unit_ids


def unit_has_valid_chunk(corpus, unit_id, min_chars=80):
    chunks = corpus["chunks_by_provision"].get(unit_id, [])
    return any(c.get("chunk_id") and len((c.get("chunk_text") or "").strip()) >= min_chars for c in chunks)

sampled_doc_ids, sampled_unit_ids_raw = sample_documents_then_provisions(
    corpus,
    CONFIG.get("sampled_provisions_target", 1200),
    CONFIG.get("min_provisions_per_sampled_doc", 2),
    CONFIG.get("max_provisions_per_sampled_doc", 5),
)
loaded_seed_chunks = hydrate_chunks_for_units(corpus, sampled_unit_ids_raw)
seed_unit_ids = [uid for uid in sampled_unit_ids_raw
                 if unit_has_valid_chunk(corpus, uid, CONFIG.get("min_chunk_chars_for_seed", 80))]
seed_doc_ids = list(dict.fromkeys(corpus["provisions_by_id"].get(uid, {}).get("id_str")
                                  for uid in seed_unit_ids
                                  if corpus["provisions_by_id"].get(uid, {}).get("id_str")))

avg_provisions_per_doc = len(seed_unit_ids) / max(1, len(seed_doc_ids))
sampling_stats = {
    "sampled_documents": len(seed_doc_ids),
    "total_documents": len(corpus["documents_by_id"]),
    "sampled_document_coverage_ratio": len(seed_doc_ids) / max(1, len(corpus["documents_by_id"])),
    "raw_sampled_provisions": len(sampled_unit_ids_raw),
    "sampled_provisions_with_valid_chunks": len(seed_unit_ids),
    "total_provisions": len(corpus["provisions_by_id"]),
    "sampled_provision_coverage_ratio": len(seed_unit_ids) / max(1, len(corpus["provisions_by_id"])),
    "loaded_seed_chunks": loaded_seed_chunks,
    "average_provisions_per_sampled_document": avg_provisions_per_doc,
    "document_metadata_coverage": {
        "with_edges": sum(1 for d in seed_doc_ids if corpus["edges_by_src_doc"].get(d) or corpus.get("edges_by_dst_doc", {}).get(d)),
        "with_validity": sum(1 for d in seed_doc_ids if corpus["validity_by_doc"].get(d)),
        "text_available": sum(1 for d in seed_doc_ids if corpus["provenance_by_doc"].get(d, {}).get("text_status") == "available"),
    },
}
print("Sampling stats:")
print(f"  Sampled documents: {sampling_stats['sampled_documents']} / {sampling_stats['total_documents']} ({sampling_stats['sampled_document_coverage_ratio']:.4%})")
print(f"  Raw sampled provisions: {sampling_stats['raw_sampled_provisions']}")
print(f"  Sampled provisions with valid chunks: {sampling_stats['sampled_provisions_with_valid_chunks']} / {sampling_stats['total_provisions']} ({sampling_stats['sampled_provision_coverage_ratio']:.4%})")
print(f"  Loaded seed chunks: {sampling_stats['loaded_seed_chunks']}")
print(f"  Average provisions per sampled document: {sampling_stats['average_provisions_per_sampled_document']:.2f}")
print("  Document metadata coverage:", sampling_stats["document_metadata_coverage"])


## 6. Related Bundle Construction

In [ ]:
# rel_canonical values semantic enough to support graph/cross-document questions.
SEMANTIC_REL_TYPES = ("rel_amends", "rel_replaces", "rel_refers_to", "rel_conditions_on")
CATEGORY_ORDER = ["single_hop", "citation", "multi_hop", "cross_document", "legal_validity", "unanswerable"]


def first_provision_id_for_doc(corpus, doc_id):
    unit_ids = corpus["provision_ids_by_doc"].get(doc_id, [])
    return unit_ids[0] if unit_ids else None


def doc_title(corpus, doc_id):
    return corpus["documents_by_id"].get(doc_id, {}).get("title") or doc_id


def classify_bundle(unit_ids, edges_used, corpus):
    doc_ids = list(dict.fromkeys(
        corpus["provisions_by_id"].get(uid, {}).get("id_str") for uid in unit_ids
        if corpus["provisions_by_id"].get(uid, {}).get("id_str")
    ))
    categories = ["single_hop", "unanswerable"]
    if any(corpus["provisions_by_id"].get(uid, {}).get("coverage_verified") for uid in unit_ids):
        categories.append("citation")
    if len(unit_ids) >= 2 and len(set(doc_ids)) == 1:
        categories.append("multi_hop")
    if len(set(doc_ids)) >= 2 or edges_used:
        categories.append("cross_document")
    if any(doc_id in corpus["validity_by_doc"] for doc_id in doc_ids):
        categories.append("legal_validity")
    return [cat for cat in CATEGORY_ORDER if cat in set(categories)]


def choose_sibling_units(corpus, doc_id, seed_unit_id, limit):
    siblings = [uid for uid in corpus["provision_ids_by_doc"].get(doc_id, []) if uid != seed_unit_id]
    if not siblings:
        return []
    try:
        pos = corpus["provision_ids_by_doc"][doc_id].index(seed_unit_id)
    except ValueError:
        pos = 0
    window = []
    for offset in range(1, limit + 2):
        for idx in (pos - offset, pos + offset):
            if 0 <= idx < len(corpus["provision_ids_by_doc"][doc_id]):
                uid = corpus["provision_ids_by_doc"][doc_id][idx]
                if uid != seed_unit_id and uid not in window:
                    window.append(uid)
            if len(window) >= limit:
                return window
    return window[:limit]


def build_related_bundle(seed_unit_id, corpus, config):
    provision = corpus["provisions_by_id"].get(seed_unit_id)
    if not provision:
        return None
    seed_doc = provision.get("id_str")
    if not seed_doc:
        return None

    max_units = int(config.get("context_units_per_task", 5) or 5)
    unit_ids = [seed_unit_id]
    edges_used = []

    # 1) Document structure: nearby provisions from the same document support multi-hop.
    for uid in choose_sibling_units(corpus, seed_doc, seed_unit_id, max_units - len(unit_ids)):
        if uid not in unit_ids:
            unit_ids.append(uid)
        if len(unit_ids) >= max_units:
            break

    # 2) Graph edges: linked documents support cross-document/legal lineage contexts.
    linked_edges = [e for e in corpus["edges_by_src_doc"].get(seed_doc, [])
                    if e.get("dst_id") and not e.get("external_target")]
    linked_edges += [e for e in corpus.get("edges_by_dst_doc", {}).get(seed_doc, [])
                     if e.get("src_id") and not e.get("external_target")]
    for edge in linked_edges[: max_units * 2]:
        other_doc = edge.get("dst_id") if edge.get("src_id") == seed_doc else edge.get("src_id")
        related_unit = first_provision_id_for_doc(corpus, other_doc)
        if related_unit and related_unit not in unit_ids:
            unit_ids.append(related_unit)
            edges_used.append(edge)
        if len(unit_ids) >= max_units:
            break

    # 3) Validity timeline: add counterparty docs when present in document-level events.
    for event in corpus["validity_by_doc"].get(seed_doc, [])[: max_units * 2]:
        counterparty = event.get("counterparty_id") or event.get("related_id") or event.get("target_id")
        related_unit = first_provision_id_for_doc(corpus, counterparty)
        if related_unit and related_unit not in unit_ids:
            unit_ids.append(related_unit)
        if len(unit_ids) >= max_units:
            break

    unit_ids = unit_ids[:max_units]
    doc_ids = list(dict.fromkeys(corpus["provisions_by_id"].get(uid, {}).get("id_str") for uid in unit_ids
                                 if corpus["provisions_by_id"].get(uid, {}).get("id_str")))
    categories = classify_bundle(unit_ids, edges_used, corpus)
    if len(unit_ids) < 2 or len(categories) < 2:
        return None

    bundle_id = "bundle-" + hashlib.sha1(("|".join(unit_ids) + "|" + ",".join(categories)).encode()).hexdigest()[:12]
    return {
        "bundle_id": bundle_id,
        "seed_unit_id": seed_unit_id,
        "unit_ids": unit_ids,
        "document_ids": doc_ids,
        "categories": categories,
        "edges_used": edges_used[:4],
    }


def build_generation_plan(bundle, corpus, config):
    max_items = int(config.get("batch_size", 6) or 6)
    max_per_center = int(config.get("max_qa_per_center_in_request", 2) or 2)
    center_counts = Counter()
    plan = []

    def add_item(category, center_uid, evidence_uids, doc_ids=None, edges=None):
        if len(plan) >= max_items or center_counts[center_uid] >= max_per_center:
            return
        if category in [p["category"] for p in plan] and len(set(bundle["categories"])) >= max_items:
            return
        plan_id = f"p{len(plan) + 1}"
        evidence_uids = [] if category == "unanswerable" else list(dict.fromkeys(evidence_uids))
        doc_ids = [] if category == "unanswerable" else list(dict.fromkeys(
            doc_ids or [corpus["provisions_by_id"].get(uid, {}).get("id_str") for uid in evidence_uids
                        if corpus["provisions_by_id"].get(uid, {}).get("id_str")]
        ))
        plan.append({
            "plan_id": plan_id,
            "category": category,
            "center_unit_id": center_uid,
            "evidence_unit_ids": evidence_uids,
            "document_ids": doc_ids,
            "edges_used": [] if category == "unanswerable" else (edges or []),
        })
        center_counts[center_uid] += 1

    units = bundle["unit_ids"]
    seed = bundle["seed_unit_id"]
    same_doc_units = [uid for uid in units if corpus["provisions_by_id"].get(uid, {}).get("id_str") ==
                      corpus["provisions_by_id"].get(seed, {}).get("id_str")]

    if "single_hop" in bundle["categories"]:
        add_item("single_hop", seed, [seed])
    if "citation" in bundle["categories"]:
        citation_uid = next((uid for uid in units if corpus["provisions_by_id"].get(uid, {}).get("coverage_verified")), seed)
        add_item("citation", citation_uid, [citation_uid])
    if "multi_hop" in bundle["categories"] and len(same_doc_units) >= 2:
        add_item("multi_hop", same_doc_units[0], same_doc_units[:3])
    if "cross_document" in bundle["categories"]:
        add_item("cross_document", seed, units[:], edges=bundle.get("edges_used", []))
    if "legal_validity" in bundle["categories"]:
        validity_uid = next((uid for uid in units if corpus["provisions_by_id"].get(uid, {}).get("id_str") in corpus["validity_by_doc"]), seed)
        related_doc = corpus["provisions_by_id"].get(validity_uid, {}).get("id_str")
        validity_units = [uid for uid in units if corpus["provisions_by_id"].get(uid, {}).get("id_str") == related_doc] or [validity_uid]
        add_item("legal_validity", validity_uid, validity_units + [uid for uid in units if uid not in validity_units][:2], edges=bundle.get("edges_used", []))
    if "unanswerable" in bundle["categories"]:
        add_item("unanswerable", seed, [], doc_ids=[])

    # Fill remaining slots with category rotation over different centers, still respecting max_per_center.
    for category in bundle["categories"]:
        for uid in units:
            if len(plan) >= max_items:
                break
            if category == "unanswerable":
                continue
            if category == "multi_hop":
                doc_id = corpus["provisions_by_id"].get(uid, {}).get("id_str")
                evidence = [x for x in units if corpus["provisions_by_id"].get(x, {}).get("id_str") == doc_id][:3]
            elif category == "cross_document":
                evidence = units[:]
            else:
                evidence = [uid]
            add_item(category, uid, evidence, edges=bundle.get("edges_used", []))
        if len(plan) >= max_items:
            break
    return plan[:max_items]


def build_context_bundles(seed_unit_ids, corpus, config):
    bundles = []
    seen_bundles = set()
    bundles_per_seed_doc = Counter()
    target = math.ceil((config.get("main_target_total", config["target_total"]) * config["oversample_factor"]) / config.get("batch_size", 6))
    max_per_seed_doc = int(config.get("max_bundles_per_seed_document", 1) or 1)

    def try_add_bundle(seed_unit_id, enforce_doc_cap=True):
        seed_doc = corpus["provisions_by_id"].get(seed_unit_id, {}).get("id_str")
        if not seed_doc:
            return False
        if enforce_doc_cap and bundles_per_seed_doc[seed_doc] >= max_per_seed_doc:
            return False
        bundle = build_related_bundle(seed_unit_id, corpus, config)
        if not bundle or bundle["bundle_id"] in seen_bundles:
            return False
        plan = build_generation_plan(bundle, corpus, config)
        if len(plan) < 3:
            return False
        bundle["plan"] = plan
        bundles.append(bundle)
        seen_bundles.add(bundle["bundle_id"])
        bundles_per_seed_doc[seed_doc] += 1
        return True

    # Pass 1: maximize document coverage by allowing only N bundles per seed document.
    for seed_unit_id in seed_unit_ids:
        try_add_bundle(seed_unit_id, enforce_doc_cap=True)
        if len(bundles) >= target:
            break

    # Pass 2: if strict doc coverage cannot fill the target, relax the per-doc cap.
    if len(bundles) < target:
        for seed_unit_id in seed_unit_ids:
            try_add_bundle(seed_unit_id, enforce_doc_cap=False)
            if len(bundles) >= target:
                break

    return bundles

context_bundles = build_context_bundles(seed_unit_ids, corpus, CONFIG)
context_unit_ids = {uid for bundle in context_bundles for uid in bundle["unit_ids"]}
loaded_chunks = hydrate_chunks_for_units(corpus, context_unit_ids)

# Drop answerable plan slots whose evidence cannot resolve to loaded chunks.
for bundle in context_bundles:
    valid_plan = []
    for item in bundle.get("plan", []):
        if item.get("category") == "unanswerable":
            valid_plan.append(item)
            continue
        evidence = item.get("evidence_unit_ids", [])
        if evidence and all(unit_has_valid_chunk(corpus, uid, CONFIG.get("min_chunk_chars_for_seed", 80)) for uid in evidence):
            valid_plan.append(item)
    bundle["plan"] = valid_plan
context_bundles = [b for b in context_bundles if len(b.get("plan", [])) >= 3]

plan_category_counts = Counter(item["category"] for bundle in context_bundles for item in bundle["plan"])
bundle_doc_ids = {doc_id for bundle in context_bundles for doc_id in bundle.get("document_ids", [])}
bundle_seed_doc_ids = {corpus["provisions_by_id"].get(bundle.get("seed_unit_id"), {}).get("id_str") for bundle in context_bundles}
bundle_seed_doc_ids = {doc_id for doc_id in bundle_seed_doc_ids if doc_id}
bundle_generation_stats = {
    "context_bundles": len(context_bundles),
    "context_provisions": len(context_unit_ids),
    "bundle_seed_documents": len(bundle_seed_doc_ids),
    "bundle_documents": len(bundle_doc_ids),
    "loaded_additional_context_chunks": loaded_chunks,
    "average_context_provisions_per_bundle": sum(len(b["unit_ids"]) for b in context_bundles) / max(1, len(context_bundles)),
    "planned_generation_categories": dict(plan_category_counts),
}
print(f"Built {bundle_generation_stats['context_bundles']} context bundles; hydrated {loaded_chunks} additional chunks for {len(context_unit_ids)} context provisions.")
print(f"Bundle seed-document coverage: {bundle_generation_stats['bundle_seed_documents']} seed docs")
print(f"Bundle total document coverage: {bundle_generation_stats['bundle_documents']} documents; avg context provisions/bundle: {bundle_generation_stats['average_context_provisions_per_bundle']:.2f}")
print("Planned generation categories:", bundle_generation_stats["planned_generation_categories"])

## 7. Prompt templates + context builder

In [ ]:
import random

PERSONAS = [
    "một người dân bình thường",
    "một hộ gia đình",
    "một doanh nghiệp nhỏ",
    "một người lao động",
    "một chủ hộ kinh doanh cá thể",
    "một cán bộ/công chức đang xử lý hồ sơ cho người dân (hỏi thay, không hỏi về quy trình nội bộ)",
]

OPENERS = [
    "Tôi...",
    "Gia đình tôi...",
    "Nếu...",
    "Trong trường hợp...",
    "Có cần...",
    "Có được...",
    "Làm sao để...",
    "Doanh nghiệp của tôi...",
    "Chúng tôi...",
    "Sắp tới tôi...",
]

TEMPLATES = {
    "single_hop": [
        "Người hỏi muốn biết mình/tổ chức mình có đủ điều kiện để làm một việc cụ thể hay không.",
        "Người hỏi muốn biết mình phải làm gì, theo trình tự nào, nộp ở đâu để hoàn tất một việc.",
        "Người hỏi muốn biết cần chuẩn bị giấy tờ/hồ sơ gì cho tình huống của mình.",
        "Người hỏi muốn biết mình có bao nhiêu thời gian để thực hiện một việc, hoặc hậu quả nếu trễ hạn.",
        "Người hỏi muốn biết mình được hưởng quyền lợi, ưu đãi, hoặc bảo vệ pháp lý nào trong tình huống của họ.",
        "Người hỏi muốn biết mình có nghĩa vụ gì phải thực hiện trong một tình huống cụ thể.",
        "Người hỏi muốn biết nếu vi phạm/không thực hiện đúng thì sẽ bị xử lý/xử phạt thế nào.",
        "Người hỏi muốn biết có trường hợp ngoại lệ nào giúp họ không phải thực hiện một yêu cầu nào đó.",
        "Người hỏi muốn biết mình phải trả bao nhiêu phí/lệ phí cho một thủ tục (chỉ dùng nếu bó ngữ cảnh có nêu số liệu).",
        "Người hỏi muốn biết có việc gì họ đang định làm mà thực ra bị cấm hoặc không thuộc diện được phép.",
    ],
    "citation": [
        "Người hỏi cần một căn cứ pháp lý rõ ràng để bảo vệ quyền lợi của mình (ví dụ để nộp cho cơ quan/tòa án), nên buộc phải hỏi đúng quy định/điều khoản nào áp dụng.",
        "Người hỏi nghe ai đó nói về 1 quy định và muốn xác minh lại xem có đúng theo văn bản/điều khoản cụ thể nào không.",
    ],
    "multi_hop": [
        "Người hỏi ở trong một tình huống có 2 yếu tố cùng lúc (ví dụ vừa thuộc nhóm A vừa làm việc B) và muốn biết mình có đủ điều kiện/bị hạn chế gì không.",
        "Người hỏi cần thực hiện một việc gồm nhiều bước, muốn biết ngoài điều kiện chính còn cần đáp ứng thêm gì nữa không.",
        "Người hỏi muốn biết trong tình huống của mình, họ vừa có quyền gì vừa phải làm nghĩa vụ gì song song.",
        "Người hỏi muốn biết trường hợp ngoại lệ của họ có áp dụng không, khi ngoại lệ đó chỉ có hiệu lực nếu thỏa đồng thời nhiều điều kiện.",
    ],
    "cross_document": [
        "Người hỏi thấy 2 nơi/2 nguồn nói khác nhau về cùng 1 vấn đề và muốn biết nên làm theo quy định nào.",
        "Người hỏi muốn biết giữa 2 tình huống liên quan, quy định yêu cầu khác nhau ra sao cho từng trường hợp.",
        "Người hỏi vừa nghe quy định thay đổi/cập nhật, muốn biết giờ phải làm theo cách nào.",
        "Người hỏi lo lắng làm theo quy định này có ảnh hưởng đến nghĩa vụ ở một việc khác không.",
    ],
    "legal_validity": [
        "Người hỏi không chắc quy định mà họ biết có còn áp dụng ở thời điểm hiện tại hay đã bị thay đổi.",
        "Người hỏi muốn biết cách làm cũ mà họ từng biết có còn đúng không, hay giờ phải làm theo cách mới.",
        "Người hỏi muốn biết quy định này có áp dụng cho đúng trường hợp/khu vực/thời điểm của họ không.",
        "Người hỏi phân vân giữa 2 quy định khác cấp độ, muốn biết nên ưu tiên theo quy định nào.",
    ],
    "unanswerable": [
        "Người hỏi hỏi một chi tiết rất cụ thể (số tiền chính xác, địa chỉ, số điện thoại, mẫu đơn...) mà bó ngữ cảnh không hề nêu.",
        "Người hỏi hỏi về hậu quả/tình huống nằm ngoài phạm vi mà bó ngữ cảnh đề cập, dù nghe có vẻ liên quan.",
        "Người hỏi hỏi về thủ tục/điều kiện mà bó ngữ cảnh chỉ lướt qua chủ đề nhưng không đủ chi tiết để trả lời chắc chắn.",
    ],
}

AVOID_QUESTION_PATTERNS = [
    "Theo Nghị định ... thì ...",
    "Theo Luật ... thì ...",
    "Theo Thông tư ... thì ...",
    "Điều ... quy định gì?",
    "Khoản ... quy định gì?",
    "Nghị định ... quy định gì?",
    "Thông tư ... quy định gì?",
    "Quyết định số ... quy định gì?",
    "Văn bản ... ban hành ngày nào?",
    "Văn bản ... căn cứ vào văn bản nào?",
    "Thông tư/Quyết định này có hiệu lực từ ngày nào?",
    "Ai ký ban hành văn bản này?",
]


def pick_from_pool(pool, used, rng):
    available = [x for x in pool if x not in used]
    return rng.choice(available or pool)


def annotate_plan_templates(plan, seed=None):
    """Gan template_hint (y dinh nguoi hoi), persona, va opener cho tung plan item.
    Xoay vong ca 3 truc (khong chi doi danh tu) de tang da dang thuc su:
    goc hoi (template_hint) x nguoi hoi (persona) x cach mo dau (opener).
    """
    rng = random.Random(seed if seed is not None else 42)
    used_hints, used_personas, used_openers = set(), set(), set()

    for item in plan:
        hint = pick_from_pool(TEMPLATES.get(item["category"], TEMPLATES["single_hop"]), used_hints, rng)
        persona = pick_from_pool(PERSONAS, used_personas, rng)
        opener = pick_from_pool(OPENERS, used_openers, rng)

        item["template_hint"] = hint
        item["persona_hint"] = persona
        item["opener_hint"] = opener

        used_hints.add(hint)
        used_personas.add(persona)
        used_openers.add(opener)

    return plan


def build_context_text(unit_ids, corpus, max_chars=5000):
    parts = []
    for uid in unit_ids:
        provision = corpus["provisions_by_id"].get(uid, {})
        chunks = corpus["chunks_by_provision"].get(uid, [])
        text = " ".join(c.get("chunk_text", "") for c in chunks)
        # provisions.jsonl dung "citation_anchor" + "path", KHONG co unit_ref/unit_heading
        header = f"[{uid}] {provision.get('citation_anchor', '')} {provision.get('path', '')}"
        parts.append(f"{header}\n{text}")
    return "\n\n".join(parts)[:max_chars]

## 8. LLM client wrapper (retry + backoff) va JSON parsing an toan

In [9]:
import re

RATE_LIMIT_STATE = {
    "last_request_ts": 0.0,
    "request_count": 0,
    "window_start": 0.0,
    "total_requests": 0,
    "lock": Lock(),
}


def _respect_rate_limit(config):
    rpm = int(config.get("requests_per_minute", 15) or 15)
    safety_factor = float(config.get("rpm_safety_factor", 0.80) or 0.80)
    effective_rpm = max(1, int(rpm * safety_factor))
    min_interval = max(float(config.get("min_request_interval_sec", 0) or 0), 60.0 / effective_rpm)
    request_cap = int(config.get("max_api_requests", 0) or 0)
    requests_per_day = int(config.get("requests_per_day", 0) or 0)
    if requests_per_day:
        request_cap = min(request_cap or requests_per_day, requests_per_day)
    window = 60.0
    state = RATE_LIMIT_STATE

    while True:
        with state["lock"]:
            now = time.time()
            if state["window_start"] == 0.0 or now - state["window_start"] >= window:
                state["window_start"] = now
                state["request_count"] = 0

            if request_cap and state["total_requests"] >= request_cap:
                raise RuntimeError(
                    f"API request budget reached: {state['total_requests']}/{request_cap}. "
                    "Increase CONFIG['max_api_requests'] only if you intentionally want more calls."
                )

            wait_for_window = 0.0
            if state["request_count"] >= effective_rpm:
                wait_for_window = window - (now - state["window_start"])

            wait_for_interval = max(0.0, state["last_request_ts"] + min_interval - now)
            wait_seconds = max(wait_for_window, wait_for_interval)
            if wait_seconds <= 0:
                state["request_count"] += 1
                state["total_requests"] += 1
                state["last_request_ts"] = now
                return

        time.sleep(wait_seconds)


def call_llm_with_retry(client, prompt, model, max_tokens, config, retry_max=None, backoff=None):
    retry_max = retry_max if retry_max is not None else config.get("retry_max", 3)
    backoff = backoff if backoff is not None else config.get("retry_backoff_sec", 5)
    last_err = None
    for attempt in range(retry_max):
        try:
            _respect_rate_limit(config)
            resp = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(max_output_tokens=max_tokens),
            )
            return resp.text or ""
        except Exception as e:
            last_err = e
            msg = str(e).lower()
            if "budget reached" in msg:
                raise
            if "429" in msg or "rate limit" in msg or "too many requests" in msg or "quota" in msg:
                if attempt < retry_max - 1:
                    time.sleep(backoff * (attempt + 1))
                    continue
            if attempt < retry_max - 1:
                time.sleep(backoff * (attempt + 1))
                continue
            raise RuntimeError(f"LLM call failed after {retry_max} retries: {last_err}")
    raise RuntimeError(f"LLM call failed after {retry_max} retries: {last_err}")


def safe_json_parse(raw):
    raw = (raw or "").strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.startswith("json"):
            raw = raw[4:]
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return None

## 9. Bundle-Level Multi-Category Generation

In [ ]:
def doc_title(corpus, doc_id):
    return corpus.get("documents_by_id", {}).get(doc_id, {}).get("title") or doc_id


GEN_PROMPT = '''Bạn là trợ lý tạo dữ liệu huấn luyện/đánh giá cho hệ thống hỏi-đáp pháp luật Việt Nam.

BỐI CẢNH QUAN TRỌNG NHẤT — đọc kỹ trước khi viết:
Người hỏi trong dữ liệu này LUÔN LÀ một người dân, hộ gia đình, doanh nghiệp, hoặc cán bộ đang xử lý một
TÌNH HUỐNG THỰC TẾ trong đời sống — họ KHÔNG đọc văn bản pháp luật, KHÔNG biết tên luật/nghị định/thông tư,
KHÔNG biết số điều/khoản. Họ chỉ biết hoàn cảnh của chính mình và muốn được hướng dẫn. Nhiệm vụ của bạn là
viết câu hỏi giống hệt như một người thật sẽ gõ vào ô hỏi-đáp pháp luật, KHÔNG PHẢI như một câu hỏi kiểm tra
kiến thức về văn bản pháp luật.

Bó ngữ cảnh (chỉ dùng thông tin trong đây, không suy diễn, không bịa, không dùng kiến thức bên ngoài):
{context}

Kế hoạch sinh câu hỏi do code tạo sẵn (mỗi dòng gồm: category, người hỏi giả định, gợi ý cách mở đầu,
và ý định thực sự của người hỏi — bám sát các gợi ý này):
{plan_text}

============================================================
NGUYÊN TẮC BẮT BUỘC VỀ PHONG CÁCH CÂU HỎI (ưu tiên cao nhất)
============================================================

1. Mặc định câu hỏi phải xuất phát từ TÌNH HUỐNG của người hỏi, không phải từ CẤU TRÚC văn bản.
   - Người hỏi không biết và không cần biết tên nghị định/luật/thông tư, số điều, số khoản.
   - Chỉ được nhắc tên/số văn bản khi category là "citation", "cross_document", hoặc "legal_validity"
     VÀ việc nhắc đó thực sự cần thiết để phân biệt/đối chiếu — không nhắc tùy tiện.

2. Câu hỏi phải xoay quanh nhu cầu thực tế, ví dụ: đủ điều kiện hay không, phải làm gì, cần hồ sơ gì,
   thời hạn bao lâu, được hưởng quyền/lợi ích gì, có nghĩa vụ gì, vi phạm thì bị xử phạt thế nào,
   có ngoại lệ nào không, quy định có còn hiệu lực không.

3. Dùng các cách mở đầu tự nhiên và ĐA DẠNG, ví dụ: "Tôi...", "Gia đình tôi...", "Nếu...",
   "Trong trường hợp...", "Có cần...", "Có được...", "Làm sao để...", "Doanh nghiệp của tôi...".
   Không dùng cùng 1 cách mở đầu cho nhiều câu hỏi liên tiếp trong cùng batch.

4. TUYỆT ĐỐI CẤM các mẫu câu hỏi document-centric/metadata-centric sau (và mọi biến thể tương đương):
   - "Theo Nghị định ... thì ...", "Theo Luật ... thì ...", "Theo Thông tư ... thì ..."
   - "Điều ... quy định gì?", "Khoản ... quy định gì?"
   - "Nghị định/Thông tư/Quyết định ... quy định gì?"
   - "Văn bản ... ban hành ngày nào?", "Văn bản ... căn cứ vào văn bản nào?"
   - "Ai ký ban hành văn bản này?", "Thông tư này có hiệu lực từ ngày nào?"
   Các câu hỏi kiểu hỏi về NGÀY HIỆU LỰC, CƠ QUAN BAN HÀNH, NGƯỜI KÝ, SỐ ĐIỀU/KHOẢN thuần túy
   (không gắn với nhu cầu thực tế nào) chỉ được chiếm tối đa **5–10% tổng số QA trong batch này**.
   Phần lớn câu hỏi phải lấy nội dung QUY ĐỊNH để trả lời, không phải hỏi về CẤU TRÚC văn bản.

============================================================
VÍ DỤ BAD -> GOOD (học phong cách, KHÔNG copy nguyên văn)
============================================================

Bad: "Nghị định 100/2019/NĐ-CP quy định gì về nồng độ cồn khi lái xe?"
Good: "Tôi uống một lon bia rồi lái xe máy về nhà thì có bị phạt không?"

Bad: "Điều 5 Luật Đất đai quy định về hạn mức giao đất như thế nào?"
Good: "Gia đình tôi được giao tối đa bao nhiêu đất nông nghiệp để canh tác?"

Bad: "Thông tư này có hiệu lực từ ngày nào?"
Good: "Quy định về việc cấp giấy phép hành nghề y mà tôi biết trước đây giờ có còn áp dụng không, hay đã đổi cách làm khác?"

Bad: "Khoản 2 Điều 10 Nghị định XYZ quy định mức phạt bao nhiêu?"
Good: "Nếu công ty tôi chậm đóng bảo hiểm xã hội cho nhân viên vài tháng thì bị xử phạt thế nào?"

Bad: "Quyết định số 22/2023/QĐ-UBND do ai ký ban hành?"
Good: "Tôi muốn xin cấp phép xây nhà ở thì cần chuẩn bị những giấy tờ gì?"

Bad (citation nhưng vẫn document-centric): "Điều nào của luật quy định về ly hôn đơn phương?"
Good (citation nhưng vẫn user-centric): "Tôi muốn ly hôn đơn phương thì cần dựa vào căn cứ pháp lý nào để tòa chấp nhận đơn của tôi?"

============================================================
YÊU CẦU BẮT BUỘC KHÁC
============================================================

- Viết toàn bộ câu hỏi, câu trả lời và giải thích bằng tiếng Việt có dấu.
- Mỗi item phải bám đúng plan_id, category, persona và opener_hint trong kế hoạch.
- Chỉ dùng thông tin có trong bó ngữ cảnh; không suy diễn, không bịa thêm, không dùng kiến thức bên ngoài.
- Không tự khai, đoán hoặc trả về bất kỳ mã document/provision/chunk nào — ground truth ID do code gán
  từ plan_id, không lấy từ output của model.
- Không tạo quá nhiều câu xoay quanh cùng một điều; kế hoạch đã giới hạn tối đa 1-2 QA cho cùng center provision.
- Với category hoặc answer_type = "unanswerable": đặt reference_answer = "Không có đủ thông tin trong đoạn luật."
  và giải thích ngắn trong answer_explanation rằng bó ngữ cảnh không nêu thông tin đó.
- Với answer_type = "boolean": reference_answer CHỈ được là một trong hai chuỗi "Có" hoặc "Không".
  Mọi diễn giải phải đặt trong answer_explanation.
- Với answer_type = "extractive": reference_answer phải là cụm/câu trích trực tiếp từ bó ngữ cảnh, ngắn gọn nhất có thể.
- Với answer_type = "abstractive": reference_answer là câu trả lời tổng hợp ngắn, mọi ý phải được hỗ trợ
  bởi bó ngữ cảnh; nêu căn cứ pháp lý (tên văn bản/điều khoản) trong answer_explanation khi phù hợp
  (căn cứ nằm ở answer_explanation, KHÔNG nằm trong câu hỏi).
- Trả về CHỈ JSON hợp lệ, không markdown, không giải thích ngoài JSON.

Định dạng JSON bắt buộc:
{{"items": [
  {{"plan_id": "p1", "category": "single_hop|citation|multi_hop|cross_document|legal_validity|unanswerable",
    "question": "...", "reference_answer": "...", "answer_explanation": "...",
    "answer_type": "extractive|abstractive|boolean|unanswerable"}},
  ...
]}}

Hãy tạo đúng {batch_size} cặp hỏi-đáp theo kế hoạch trên.
'''


def task_key(bundle):
    return bundle.get("bundle_id") or hashlib.sha1("|".join(bundle.get("unit_ids", [])).encode()).hexdigest()[:12]


def normalize_answer_item(item):
    answer_type = str(item.get("answer_type", "extractive") or "extractive").strip().lower()
    if answer_type not in {"extractive", "abstractive", "boolean", "unanswerable"}:
        answer_type = "extractive"

    reference_answer = str(item.get("reference_answer", "") or "").strip()
    answer_explanation = str(item.get("answer_explanation", "") or "").strip()

    if answer_type == "boolean":
        lowered = reference_answer.lower()
        if lowered.startswith("có") or lowered.startswith("co"):
            reference_answer = "Có"
        elif lowered.startswith("không") or lowered.startswith("khong"):
            reference_answer = "Không"
        else:
            answer_type = "abstractive"
    elif answer_type == "unanswerable":
        reference_answer = "Không có đủ thông tin trong đoạn luật."
        if not answer_explanation:
            answer_explanation = "Bó ngữ cảnh không nêu thông tin đủ để trả lời câu hỏi."

    return answer_type, reference_answer, answer_explanation


def format_bundle_context(bundle, corpus, max_chars=9000):
    doc_lines = []
    for doc_id in bundle.get("document_ids", []):
        events = corpus["validity_by_doc"].get(doc_id, [])[:3]
        event_text = "; ".join(
            f"{e.get('event_date_iso') or e.get('date')}: {e.get('event_type')}"
            for e in events if e.get("event_type")
        )
        suffix = f" | validity: {event_text}" if event_text else ""
        doc_lines.append(f"- {doc_title(corpus, doc_id)} [{doc_id}]{suffix}")

    edge_lines = []
    for e in bundle.get("edges_used", [])[:4]:
        edge_lines.append(
            f"- {doc_title(corpus, e.get('src_id'))} [{e.get('src_id')}] --{e.get('rel_canonical')}--> "
            f"{doc_title(corpus, e.get('dst_id'))} [{e.get('dst_id')}]"
        )

    unit_text = build_context_text(bundle.get("unit_ids", []), corpus, max_chars=max_chars)
    parts = [
        "Tài liệu trong bundle (chỉ để code đối chiếu — KHÔNG đưa số hiệu này vào câu hỏi trừ khi category cần):",
        "\n".join(doc_lines) or "(không có metadata tài liệu)",
    ]
    if edge_lines:
        parts.extend(["Quan hệ graph/edge đã xác minh:", "\n".join(edge_lines)])
    parts.extend(["Các điều/chunk liên quan (nội dung dùng để trả lời):", unit_text])
    return "\n\n".join(parts)[:max_chars]


def format_generation_plan(plan):
    lines = []
    for item in plan:
        evidence = ", ".join(item.get("evidence_unit_ids", [])) or "ground truth rỗng"
        lines.append(
            f"- {item['plan_id']}: category={item['category']}; "
            f"center={item.get('center_unit_id')}; evidence={evidence}; "
            f"nguoi_hoi={item.get('persona_hint', '')}; "
            f"mo_dau_goi_y={item.get('opener_hint', '')}; "
            f"y_dinh_cau_hoi={item.get('template_hint', '')}"
        )
    return "\n".join(lines)


def generate_bundle_batch(client, bundle, corpus, config):
    source_task_key = task_key(bundle)
    plan = list(bundle.get("plan", [])[: int(config.get("batch_size", 6) or 6)])
    if not plan:
        return []
    # annotate_plan_templates gio gan ca template_hint + persona_hint + opener_hint
    # (dinh nghia trong cell templates chay truoc do)
    plan = annotate_plan_templates(plan, seed=hash(source_task_key) % (2**31))
    plan_by_id = {p["plan_id"]: p for p in plan}
    context = format_bundle_context(bundle, corpus)
    prompt = GEN_PROMPT.format(
        batch_size=len(plan),
        context=context,
        plan_text=format_generation_plan(plan),
    )

    raw = call_llm_with_retry(client, prompt, config["generator_model"], config["max_tokens_gen"], config)
    parsed = safe_json_parse(raw)
    if parsed is None:
        return []

    items = parsed.get("items") or []
    results = []
    used_plan_ids = set()
    for item in items:
        plan_id = str(item.get("plan_id", "") or "").strip()
        plan_item = plan_by_id.get(plan_id)
        if not plan_item or plan_id in used_plan_ids:
            continue
        used_plan_ids.add(plan_id)

        parsed_question = str(item.get("question", "") or "").strip()
        if not parsed_question:
            continue
        answer_type, reference_answer, answer_explanation = normalize_answer_item(item)
        category = plan_item["category"]
        if category == "unanswerable":
            answer_type = "unanswerable"
            reference_answer = "Không có đủ thông tin trong đoạn luật."
            gt_doc_ids = []
            gt_provision_ids = []
            edges_used = []
        else:
            gt_doc_ids = list(plan_item.get("document_ids", []))
            gt_provision_ids = list(plan_item.get("evidence_unit_ids", []))
            edges_used = plan_item.get("edges_used", [])

        qa_id = "vlrag-" + hashlib.sha1((parsed_question + source_task_key + plan_id).encode()).hexdigest()[:10]
        results.append({
            "qa_id": qa_id,
            "question": parsed_question,
            "reference_answer": reference_answer,
            "answer_explanation": answer_explanation,
            "answer_type": answer_type,
            "ground_truth": {
                "document_ids": gt_doc_ids,
                "provision_ids": gt_provision_ids,
                "chunk_ids": [],
            },
            "edges_used": edges_used,
            "category": category,
            "difficulty": None,
            "source_type": "generated",
            "generator_model": config["generator_model"],
            "corpus_version": config["corpus_version"],
            "as_of_date": config["as_of_date"],
            "status": "candidate",
            "source_task_key": source_task_key,
            "source_bundle_id": bundle.get("bundle_id"),
            "source_plan_id": plan_id,
            "center_provision_id": plan_item.get("center_unit_id"),
            "persona_used": plan_item.get("persona_hint"),
        })
    return results

## 10. Minimal Evidence Tagging

In [ ]:
def text_overlap_ratio(answer_text, chunk_text):
    a = set(answer_text.lower().split())
    b = set(chunk_text.lower().split())
    if not a or not b:
        return 0.0
    return len(a & b) / len(a)

def tag_minimal_evidence(qa, corpus, overlap_threshold=0.25):
    ground_truth = qa.get("ground_truth", {})
    if qa.get("category") == "unanswerable" or qa.get("answer_type") == "unanswerable":
        ground_truth["document_ids"] = []
        ground_truth["provision_ids"] = []
        ground_truth["chunk_ids"] = []
        qa["ground_truth"] = ground_truth
        return qa

    # Recover document IDs from code-owned provision evidence if needed.
    provision_ids = [uid for uid in ground_truth.get("provision_ids", []) if uid in corpus["provisions_by_id"]]
    if not ground_truth.get("document_ids") and provision_ids:
        ground_truth["document_ids"] = list(dict.fromkeys(
            corpus["provisions_by_id"].get(uid, {}).get("id_str") for uid in provision_ids
            if corpus["provisions_by_id"].get(uid, {}).get("id_str")
        ))
    ground_truth["provision_ids"] = provision_ids

    chunk_ids = []
    for uid in provision_ids:
        for c in corpus["chunks_by_provision"].get(uid, []):
            if text_overlap_ratio(qa.get("reference_answer", ""), c.get("chunk_text", "")) >= overlap_threshold:
                chunk_id = c.get("chunk_id")
                if chunk_id:
                    chunk_ids.append(chunk_id)
    if not chunk_ids:
        for uid in provision_ids:
            chunk_ids.extend(c.get("chunk_id") for c in corpus["chunks_by_provision"].get(uid, []) if c.get("chunk_id"))
    ground_truth["chunk_ids"] = list(dict.fromkeys(chunk_ids))
    qa["ground_truth"] = ground_truth
    return qa


def validate_ground_truth_for_save(qa, corpus):
    ground_truth = qa.get("ground_truth", {})
    if qa.get("category") == "unanswerable" or qa.get("answer_type") == "unanswerable":
        ground_truth["document_ids"] = []
        ground_truth["provision_ids"] = []
        ground_truth["chunk_ids"] = []
        qa["ground_truth"] = ground_truth
        return True, None

    doc_ids = [d for d in ground_truth.get("document_ids", []) if d in corpus["documents_by_id"]]
    provision_ids = [uid for uid in ground_truth.get("provision_ids", []) if uid in corpus["provisions_by_id"]]
    if provision_ids and not doc_ids:
        doc_ids = list(dict.fromkeys(
            corpus["provisions_by_id"].get(uid, {}).get("id_str") for uid in provision_ids
            if corpus["provisions_by_id"].get(uid, {}).get("id_str") in corpus["documents_by_id"]
        ))
    chunk_ids = [cid for cid in ground_truth.get("chunk_ids", []) if cid]

    if not doc_ids:
        return False, "missing_document_ids"
    if not provision_ids:
        return False, "missing_provision_ids"
    if not chunk_ids:
        return False, "missing_chunk_ids"

    ground_truth["document_ids"] = list(dict.fromkeys(doc_ids))
    ground_truth["provision_ids"] = list(dict.fromkeys(provision_ids))
    ground_truth["chunk_ids"] = list(dict.fromkeys(chunk_ids))
    qa["ground_truth"] = ground_truth
    return True, None


def filter_valid_answerable_qas(qas, corpus):
    valid = []
    rejected = Counter()
    for qa in qas:
        ok, reason = validate_ground_truth_for_save(qa, corpus)
        if ok:
            valid.append(qa)
        else:
            qa["status"] = "rejected_missing_ground_truth"
            qa["ground_truth_validation_error"] = reason
            rejected[reason] += 1
    return valid, rejected

## 11. Verification (1 verifier, checklist gop)

In [12]:
VERIFY_PROMPT = '''Bạn là verifier pháp lý độc lập. Hãy đánh giá {n_items} QA dưới đây dựa duy nhất trên evidence được cung cấp.

Nguyên tắc bắt buộc:
- Viết toàn bộ nhận xét bằng tiếng Việt có dấu.
- Không dùng kiến thức bên ngoài evidence.
- Kiểm tra đủ 5 tiêu chí theo §1.2: answer correctness, evidence sufficiency, no hallucination, category correctness, difficulty correctness.
- Nếu bất kỳ tiêu chí nào FAIL thì QA phải bị loại.
- Với answer_type = "boolean", reference_answer phải đúng chính xác "Có" hoặc "Không"; phần giải thích nếu có chỉ nằm trong answer_explanation.
- Với category/answer_type = "unanswerable", câu hỏi chỉ PASS khi evidence thật sự không chứa thông tin đủ để trả lời, và ground truth rỗng là hợp lý.

{items_block}

Trả về CHỈ JSON hợp lệ, không markdown, theo đúng định dạng:
{{"items": [
  {{"qa_id": "...", "answer_correctness": "PASS|FAIL", "evidence_sufficiency": "PASS|FAIL", "no_hallucination": "PASS|FAIL", "category_correctness": "PASS|FAIL", "difficulty_correctness": "PASS|FAIL", "difficulty": "easy|medium|hard", "notes": "..."}}
]}}
'''


def verify_qas(client, qas, corpus, config):
    if not qas:
        return []
    items_block = []
    for qa in qas:
        ground_truth = qa.get("ground_truth", {})
        evidence_text = build_context_text(ground_truth.get("provision_ids", []), corpus)
        if not evidence_text and (qa.get("category") == "unanswerable" or qa.get("answer_type") == "unanswerable"):
            evidence_text = "[Ground truth rỗng theo thiết kế: câu hỏi được tạo để không trả lời được từ đoạn luật.]"
        items_block.append(
            f"QA_ID: {qa.get('qa_id')}\n"
            f"Câu hỏi: {qa.get('question', '')}\n"
            f"Câu trả lời chuẩn: {qa.get('reference_answer', '')}\n"
            f"Giải thích câu trả lời: {qa.get('answer_explanation', '')}\n"
            f"Answer type: {qa.get('answer_type', '')}\n"
            f"Category đề xuất: {qa.get('category', '')}\n"
            f"Evidence:\n{evidence_text}\n"
        )
    prompt = VERIFY_PROMPT.format(n_items=len(qas), items_block="\n\n".join(items_block))
    raw = call_llm_with_retry(client, prompt, config["verifier_model"], config["max_tokens_verify"], config)
    parsed = safe_json_parse(raw)
    if parsed is None:
        return []

    verdicts = parsed.get("items") or []
    verdict_by_id = {v.get("qa_id"): v for v in verdicts if v.get("qa_id")}
    updated = []
    required_pass_fields = [
        "answer_correctness",
        "evidence_sufficiency",
        "no_hallucination",
        "category_correctness",
        "difficulty_correctness",
    ]
    for qa in qas:
        verdict = verdict_by_id.get(qa.get("qa_id"), {})
        passed = all(verdict.get(field) == "PASS" for field in required_pass_fields)
        qa["verifier_model"] = config["verifier_model"]
        qa["verifier_result"] = "pass" if passed else "fail"
        qa["verifier_checks"] = {field: verdict.get(field) for field in required_pass_fields}
        qa["difficulty"] = verdict.get("difficulty")
        qa["verifier_notes"] = verdict.get("notes", "")
        qa["status"] = "active" if passed else "rejected"
        updated.append(qa)
    return updated

## 12. Checkpointing

Kaggle notebook co gioi han thoi gian session -- checkpoint sau moi N item de resume duoc
neu bi ngat giua chung, khong phai chay lai tu dau.

In [ ]:
def load_checkpoint(path):
    if not Path(path).exists():
        return set(), set()
    done_ids = set()
    done_task_keys = set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            qa = json.loads(line)
            qa_id = qa.get("qa_id")
            if qa_id:
                done_ids.add(qa_id)
            source_task_key = qa.get("source_task_key") or qa.get("source_bundle_id")
            if source_task_key:
                done_task_keys.add(source_task_key)
    return done_ids, done_task_keys

def append_checkpoint(path, qa):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(qa, ensure_ascii=False) + "\n")

done_ids, done_task_keys = load_checkpoint(CONFIG["checkpoint_path"])
print(f"Da co {len(done_ids)} QA trong checkpoint; {len(done_task_keys)} task da hoan tat, se bo qua khi chay lai.")

## 13. Main loop: Generate Bundle -> Minimal Evidence -> Verify -> Checkpoint

Mỗi task là một related context bundle gồm 3-5 provision/chunk, sinh 5-6 QA đa category trong một request. Verifier vẫn đánh giá cả batch như trước.


In [ ]:
tasks = [bundle for bundle in context_bundles if task_key(bundle) not in done_task_keys]
expected_candidates = sum(len(bundle.get("plan", [])) for bundle in tasks)
expected_api_calls = len(tasks) * 2
effective_rpm = max(1, int(CONFIG["requests_per_minute"] * CONFIG.get("rpm_safety_factor", 1.0)))
estimated_minutes = expected_api_calls / effective_rpm if effective_rpm else 0
print(
    f"So bundle task can chay: {len(tasks)} (~{expected_candidates} QA candidates, "
    f"~{expected_api_calls} API calls before retries, ~{estimated_minutes:.1f} minutes at {effective_rpm} RPM)"
)


def process_one(bundle):
    qas = generate_bundle_batch(client, bundle, corpus, CONFIG)
    if not qas:
        return [], [], Counter()
    candidates = [dict(qa) for qa in qas]
    tagged = [tag_minimal_evidence(qa, corpus) for qa in qas]
    valid_tagged, rejected = filter_valid_answerable_qas(tagged, corpus)
    if not valid_tagged:
        return candidates, [], rejected
    verified = verify_qas(client, valid_tagged, corpus, CONFIG)
    verified_valid, verify_rejected = filter_valid_answerable_qas(verified, corpus)
    rejected.update(verify_rejected)
    return candidates, verified_valid, rejected

results = []
new_candidates = []
ground_truth_rejections = Counter()
with ThreadPoolExecutor(max_workers=CONFIG.get("max_workers", 1)) as executor:
    futures = {executor.submit(process_one, bundle): bundle for bundle in tasks}
    for i, future in enumerate(tqdm(as_completed(futures), total=len(futures))):
        try:
            candidates, qas, rejected = future.result()
        except Exception as e:
            print(f"Task failed: {e}")
            continue
        new_candidates.extend(candidates)
        ground_truth_rejections.update(rejected)
        if not qas:
            continue
        for qa in qas:
            ok, reason = validate_ground_truth_for_save(qa, corpus)
            if not ok:
                ground_truth_rejections[reason] += 1
                continue
            append_checkpoint(CONFIG["checkpoint_path"], qa)
            results.append(qa)

main_generation_stats = {
    "bundle_tasks_run": len(tasks),
    "expected_candidates": expected_candidates,
    "raw_generated": len(new_candidates),
    "verified_records_appended": len(results),
    "raw_generated_by_category": dict(Counter(qa.get("category") for qa in new_candidates)),
    "verified_records_by_category": dict(Counter(qa.get("category") for qa in results)),
    "ground_truth_validation_rejections": dict(ground_truth_rejections),
    "api_requests_used_after_main": RATE_LIMIT_STATE["total_requests"],
}
print(f"Hoan tat vong generate+verify. Tong candidate moi: {len(results)} verified records; raw generated: {len(new_candidates)}")
print("Generated QA by category:", main_generation_stats["raw_generated_by_category"])
print("Ground-truth validation rejections:", main_generation_stats["ground_truth_validation_rejections"])
print(f"API requests used this session: {RATE_LIMIT_STATE['total_requests']} / {CONFIG['max_api_requests']}")

## 14. Integrated 100 Nuance QA Generation

Sinh thêm 100 QA đặc thù pháp lý Việt Nam trong cùng pipeline/checkpoint, dùng chung corpus, verifier, rate limiter và validation.


In [ ]:
NUANCE_PROMPT = '''Bạn là trợ lý tạo dữ liệu huấn luyện/đánh giá cho hệ thống hỏi-đáp pháp luật Việt Nam.

BỐI CẢNH QUAN TRỌNG NHẤT — đọc kỹ trước khi viết:
Người hỏi trong dữ liệu này LUÔN LÀ một người dân, hộ gia đình, doanh nghiệp, hoặc cán bộ đang xử lý một
TÌNH HUỐNG THỰC TẾ — họ KHÔNG đọc văn bản pháp luật, KHÔNG biết tên luật/nghị định/thông tư, KHÔNG biết
số điều/khoản. Họ chỉ biết hoàn cảnh của mình và muốn được hướng dẫn. Viết câu hỏi giống hệt một người
thật sẽ gõ vào ô hỏi-đáp pháp luật, KHÔNG PHẢI như câu hỏi kiểm tra kiến thức về văn bản.

Nuance tag: {nuance_tag}
Category benchmark: {category}
Yêu cầu nghiệp vụ: {instruction}
Người hỏi giả định: {persona_hint}
Gợi ý cách mở đầu: {opener_hint}

Evidence/code-owned context (chỉ dùng thông tin trong đây, không suy diễn, không bịa, không dùng kiến thức bên ngoài):
{context}

============================================================
NGUYÊN TẮC BẮT BUỘC VỀ PHONG CÁCH CÂU HỎI (ưu tiên cao nhất)
============================================================

1. Câu hỏi phải xuất phát từ TÌNH HUỐNG của người hỏi, không phải từ CẤU TRÚC văn bản.
   - KHÔNG nhắc tên văn bản (Luật, Nghị định, Thông tư, Quyết định...) hoặc số điều/khoản trong câu hỏi,
     TRỪ KHI category = "citation" và việc nhắc đó thực sự cần thiết để phân biệt/đối chiếu.
   - Tên văn bản, số điều/khoản, trích dẫn chính xác luôn đặt trong `reference_answer`/`answer_explanation`,
     KHÔNG đặt trong câu hỏi.

2. Với category "legal_validity"/"cross_document" (nuance tag như continuous_amendment, partial_validity,
   multi_tier_relation, conflict_resolution): hỏi theo hướng người áp dụng thực tế đang phân vân —
   ví dụ "cách làm tôi từng biết có còn đúng không", "quy định nào tôi nên theo khi 2 nơi nói khác nhau",
   "phần nào vẫn áp dụng cho trường hợp của tôi" — KHÔNG hỏi kiểu "văn bản X đã bị thay thế bởi văn bản
   nào?" (đó vẫn là câu hỏi document-centric dù đúng category).

3. Dùng cách mở đầu tự nhiên theo gợi ý ở trên (`{opener_hint}`) hoặc biến thể tương đương —
   không lặp lại y hệt cấu trúc câu mẫu.

4. TUYỆT ĐỐI CẤM các mẫu sau (và mọi biến thể tương đương):
   - "Nghị định X quy định gì?", "Văn bản X ban hành ngày nào?", "Điều Y quy định gì?",
     "Văn bản X căn cứ vào văn bản nào?", "Văn bản X đã bị thay thế bởi văn bản nào?"
   - "Ai ký ban hành văn bản này?", "Thông tư này có hiệu lực từ ngày nào?"

============================================================
VÍ DỤ BAD -> GOOD theo đúng nhóm nuance (học phong cách, KHÔNG copy nguyên văn)
============================================================

[continuous_amendment / legal_validity]
Bad: "Nghị định A đã bị sửa đổi bởi nghị định nào và từ khi nào?"
Good: "Cách tôi từng làm thủ tục này 2 năm trước giờ có còn đúng không, hay đã đổi cách khác rồi?"

[partial_validity / legal_validity]
Bad: "Phần nào của Thông tư B đã hết hiệu lực?"
Good: "Trường hợp của tôi có còn áp dụng theo cách cũ không, hay chỉ một phần quy định đó đã thay đổi?"

[multi_tier_relation / cross_document]
Bad: "Nghị định C dẫn chiếu đến những văn bản nào?"
Good: "Tôi đang làm hồ sơ liên quan đến cả 2 việc này cùng lúc, vậy tôi cần đáp ứng đủ điều kiện của cả hai hay chỉ một bên?"

[conflict_resolution / legal_validity]
Bad: "Giữa văn bản D và văn bản E, văn bản nào có hiệu lực pháp lý cao hơn?"
Good: "Tôi nghe 2 nơi hướng dẫn khác nhau cho cùng một việc, vậy tôi nên làm theo hướng dẫn nào để không bị sai?"

[citation]
Bad: "Điều nào quy định về việc này?"
Good: "Tôi muốn khiếu nại quyết định này thì cần dựa vào căn cứ pháp lý nào để được xem xét?"

============================================================
YÊU CẦU BẮT BUỘC KHÁC
============================================================

- Viết tiếng Việt có dấu, tự nhiên, không yêu cầu người hỏi biết trước tên văn bản.
- Chỉ dùng thông tin trong evidence/code-owned context; không suy diễn hoặc bịa thêm.
- Không trả về document/provision/chunk ID — ground truth ID do code gán.
- Nếu `answer_type` = "boolean": `reference_answer` chỉ là "Có" hoặc "Không"; giải thích để trong `answer_explanation`.
- Nếu `answer_type` = "unanswerable": `reference_answer` = "Không có đủ thông tin trong đoạn luật.".
- Trả về CHỈ JSON hợp lệ, không markdown, không giải thích ngoài JSON:
{{"question":"...", "reference_answer":"...", "answer_explanation":"...", "answer_type":"extractive|abstractive|boolean|unanswerable"}}
'''


def first_valid_provision_for_doc(corpus, doc_id, min_chars=80):
    for uid in corpus["provision_ids_by_doc"].get(doc_id, []):
        if unit_has_valid_chunk(corpus, uid, min_chars):
            return uid
    fallback = first_provision_id_for_doc(corpus, doc_id)
    return fallback


def pick_persona_opener(seed_key):
    """Chon persona/opener xoay vong, dung lai PERSONAS/OPENERS da dinh nghia
    o cell templates truoc do (khong dinh nghia lai o day)."""
    rng = random.Random(hash(seed_key) % (2**31))
    return rng.choice(PERSONAS), rng.choice(OPENERS)


def make_nuance_task(nuance_tag, category, instruction, evidence_unit_ids, corpus, edges_used=None, extra_facts=None):
    evidence_unit_ids = list(dict.fromkeys(uid for uid in evidence_unit_ids if uid and uid in corpus["provisions_by_id"]))
    doc_ids = list(dict.fromkeys(
        corpus["provisions_by_id"].get(uid, {}).get("id_str") for uid in evidence_unit_ids
        if corpus["provisions_by_id"].get(uid, {}).get("id_str")
    ))
    if not evidence_unit_ids or not doc_ids:
        return None
    key = "|".join([nuance_tag, category, *evidence_unit_ids, json.dumps(extra_facts or {}, sort_keys=True, ensure_ascii=False)])
    task_id = "nuance-" + hashlib.sha1(key.encode()).hexdigest()[:12]
    persona_hint, opener_hint = pick_persona_opener(task_id)
    return {
        "task_id": task_id,
        "nuance_tag": nuance_tag,
        "category": category,
        "instruction": instruction,
        "evidence_unit_ids": evidence_unit_ids,
        "document_ids": doc_ids,
        "edges_used": edges_used or [],
        "extra_facts": extra_facts or {},
        "persona_hint": persona_hint,
        "opener_hint": opener_hint,
    }


def sample_continuous_amendment_tasks(corpus, n_target, seed):
    rng = random.Random(seed)
    candidates = []
    for doc_id, events in corpus["validity_by_doc"].items():
        change_events = [e for e in events if any(tok in str(e.get("event_type", "")).lower()
                         for tok in ["amend", "replace", "expire", "suspend", "partial"])]
        if len(change_events) < 2:
            continue
        uid = first_valid_provision_for_doc(corpus, doc_id, CONFIG.get("min_chunk_chars_for_seed", 80))
        if not uid:
            continue
        related = []
        for e in change_events[:4]:
            cp = e.get("counterparty_id") or e.get("related_id") or e.get("target_id")
            cp_uid = first_valid_provision_for_doc(corpus, cp, CONFIG.get("min_chunk_chars_for_seed", 80))
            if cp_uid:
                related.append(cp_uid)
        task = make_nuance_task(
            "continuous_amendment", "legal_validity",
            "Tạo câu hỏi thực tế: người áp dụng muốn biết cách làm/quy định họ từng biết có còn đúng không, "
            "hay đã đổi khác, và nên làm theo cách nào ở thời điểm hiện tại — không hỏi tên văn bản sửa đổi.",
            [uid] + related[:3], corpus,
            extra_facts={"events": change_events[:5], "doc_id": doc_id},
        )
        if task:
            candidates.append(task)
    rng.shuffle(candidates)
    return candidates[:n_target]


def sample_partial_validity_tasks(corpus, n_target, seed):
    rng = random.Random(seed)
    candidates = []
    for doc_id, events in corpus["validity_by_doc"].items():
        partial_events = [e for e in events if "partial" in str(e.get("event_type", "")).lower()
                          or str(e.get("scope", "")).lower() not in ("", "whole", "document")]
        if not partial_events:
            continue
        uid = first_valid_provision_for_doc(corpus, doc_id, CONFIG.get("min_chunk_chars_for_seed", 80))
        if not uid:
            continue
        e = partial_events[0]
        cp = e.get("counterparty_id") or e.get("related_id") or e.get("target_id")
        cp_uid = first_valid_provision_for_doc(corpus, cp, CONFIG.get("min_chunk_chars_for_seed", 80))
        task = make_nuance_task(
            "partial_validity", "legal_validity",
            "Tạo câu hỏi thực tế: người áp dụng muốn biết trường hợp/đối tượng của họ có còn được áp dụng "
            "quy định này không, hay chỉ một phần đã thay đổi — hỏi theo tình huống cụ thể, không hỏi cấu trúc văn bản.",
            [uid] + ([cp_uid] if cp_uid else []), corpus,
            extra_facts={"event": e, "doc_id": doc_id},
        )
        if task:
            candidates.append(task)
    rng.shuffle(candidates)
    return candidates[:n_target]


def sample_multi_tier_relation_tasks(corpus, n_target, seed):
    rng = random.Random(seed)
    candidates = []
    rel_types = {"rel_amends", "rel_replaces", "rel_refers_to", "rel_conditions_on"}
    for center_doc, edges in corpus["edges_by_src_doc"].items():
        rel_edges = [e for e in edges if e.get("rel_canonical") in rel_types and not e.get("external_target")]
        if len(rel_edges) < 2:
            continue
        center_uid = first_valid_provision_for_doc(corpus, center_doc, CONFIG.get("min_chunk_chars_for_seed", 80))
        related_uids = [first_valid_provision_for_doc(corpus, e.get("dst_id"), CONFIG.get("min_chunk_chars_for_seed", 80)) for e in rel_edges[:4]]
        task = make_nuance_task(
            "multi_tier_relation", "cross_document",
            "Tạo câu hỏi thực tế: người áp dụng đang ở tình huống liên quan đến nhiều quy định chồng lấn/viện dẫn "
            "lẫn nhau, muốn biết mình phải đáp ứng điều kiện của quy định nào — hỏi theo tình huống, không nhắc "
            "tên văn bản trừ khi thật cần thiết.",
            [center_uid] + [u for u in related_uids if u], corpus,
            edges_used=rel_edges[:4], extra_facts={"center_doc": center_doc, "relation_count": len(rel_edges)},
        )
        if task:
            candidates.append(task)
    rng.shuffle(candidates)
    return candidates[:n_target]


def sample_conflict_resolution_tasks(corpus, n_target, seed):
    rng = random.Random(seed)
    candidates = []
    seen = set()
    for src_doc, edges in corpus["edges_by_src_doc"].items():
        d1 = corpus["documents_by_id"].get(src_doc, {})
        r1 = d1.get("legal_authority_rank")
        if r1 is None:
            continue
        for e in edges:
            dst_doc = e.get("dst_id")
            if not dst_doc or e.get("external_target"):
                continue
            pair = tuple(sorted([src_doc, dst_doc]))
            if pair in seen:
                continue
            d2 = corpus["documents_by_id"].get(dst_doc, {})
            r2 = d2.get("legal_authority_rank")
            if r2 is None or r1 == r2:
                continue
            uid1 = first_valid_provision_for_doc(corpus, src_doc, CONFIG.get("min_chunk_chars_for_seed", 80))
            uid2 = first_valid_provision_for_doc(corpus, dst_doc, CONFIG.get("min_chunk_chars_for_seed", 80))
            winner = src_doc if r1 < r2 else dst_doc
            task = make_nuance_task(
                "conflict_resolution", "legal_validity",
                "Tạo câu hỏi thực tế: người áp dụng nghe 2 nơi/2 quy định hướng dẫn khác nhau cho cùng một việc, "
                "muốn biết nên làm theo hướng dẫn nào để không bị sai — không hỏi trực tiếp thứ bậc pháp lý.",
                [uid1, uid2], corpus, edges_used=[e],
                extra_facts={"src_rank": r1, "dst_rank": r2, "winner_doc_id": winner},
            )
            if task:
                candidates.append(task)
                seen.add(pair)
    rng.shuffle(candidates)
    return candidates[:n_target]


def sample_buffer_nuance_tasks(corpus, n_target, seed):
    rng = random.Random(seed)
    units = list(seed_unit_ids)
    rng.shuffle(units)
    tasks = []
    for i, uid in enumerate(units):
        if not unit_has_valid_chunk(corpus, uid, CONFIG.get("min_chunk_chars_for_seed", 80)):
            continue
        category = "citation" if i % 3 == 0 else "single_hop"
        buffer_hints = {
            "single_hop": "Tạo câu hỏi thực tế về điều kiện, quyền, nghĩa vụ, thủ tục, hồ sơ, thời hạn, ngoại lệ, "
                          "chế tài hoặc phí — xuất phát từ tình huống người hỏi, không hỏi metadata văn bản.",
            "citation": "Người hỏi cần một căn cứ pháp lý cụ thể để bảo vệ quyền lợi của mình (ví dụ để nộp cho "
                        "cơ quan/tòa án) — câu hỏi được phép nhắc đến việc cần 'căn cứ pháp lý' nhưng vẫn xuất "
                        "phát từ tình huống thực tế, không hỏi trống kiểu 'điều nào quy định'.",
        }
        task = make_nuance_task(
            "buffer_" + category, category,
            buffer_hints[category],
            [uid], corpus,
        )
        if task:
            tasks.append(task)
        if len(tasks) >= n_target:
            break
    return tasks


def build_nuance_tasks(corpus, config):
    quota = config.get("nuance_quota", {})
    oversample = config.get("nuance_oversample_factor", 1.2)
    seed = 137
    builders = {
        "continuous_amendment": sample_continuous_amendment_tasks,
        "partial_validity": sample_partial_validity_tasks,
        "multi_tier_relation": sample_multi_tier_relation_tasks,
        "conflict_resolution": sample_conflict_resolution_tasks,
        "buffer_single_citation": sample_buffer_nuance_tasks,
    }
    tasks = []
    for key, builder in builders.items():
        tasks.extend(builder(corpus, math.ceil(quota.get(key, 0) * oversample), seed))
    seen = set()
    deduped = []
    for task in tasks:
        if task["task_id"] in seen:
            continue
        seen.add(task["task_id"])
        deduped.append(task)
    return deduped


def format_nuance_context(task, corpus, max_chars=8000):
    fact_text = json.dumps(task.get("extra_facts", {}), ensure_ascii=False, indent=2, default=str)
    edge_lines = []
    for e in task.get("edges_used", []):
        edge_lines.append(f"- {doc_title(corpus, e.get('src_id'))} [{e.get('src_id')}] --{e.get('rel_canonical')}--> {doc_title(corpus, e.get('dst_id'))} [{e.get('dst_id')}]")
    parts = [
        "Evidence provisions (nội dung dùng để trả lời):",
        build_context_text(task.get("evidence_unit_ids", []), corpus, max_chars=max_chars),
        "Verified edges (chỉ dùng nội bộ để suy luận, KHÔNG đưa số hiệu văn bản vào câu hỏi trừ khi category cần):",
        "\n".join(edge_lines) or "(không có edge dùng trực tiếp)",
        "Code-owned facts (chỉ dùng nội bộ để chấm điểm/gán ground truth, KHÔNG đưa vào câu hỏi):",
        fact_text,
    ]
    return "\n\n".join(parts)[:max_chars]


def generate_nuance_qa(client, task, corpus, config):
    prompt = NUANCE_PROMPT.format(
        nuance_tag=task["nuance_tag"],
        category=task["category"],
        instruction=task["instruction"],
        persona_hint=task.get("persona_hint", ""),
        opener_hint=task.get("opener_hint", ""),
        context=format_nuance_context(task, corpus),
    )
    raw = call_llm_with_retry(client, prompt, config["generator_model"], config["max_tokens_gen"], config)
    parsed = safe_json_parse(raw)
    if parsed is None:
        return None
    answer_type, reference_answer, answer_explanation = normalize_answer_item(parsed)
    qa_id = "vlrag-nuance-" + hashlib.sha1((parsed.get("question", "") + task["task_id"]).encode()).hexdigest()[:10]
    return {
        "qa_id": qa_id,
        "question": str(parsed.get("question", "") or "").strip(),
        "reference_answer": reference_answer,
        "answer_explanation": answer_explanation,
        "answer_type": answer_type,
        "ground_truth": {
            "document_ids": task["document_ids"],
            "provision_ids": task["evidence_unit_ids"],
            "chunk_ids": [],
        },
        "edges_used": task.get("edges_used", []),
        "category": task["category"],
        "difficulty": None,
        "source_type": "generated",
        "generator_model": config["generator_model"],
        "corpus_version": config["corpus_version"],
        "as_of_date": config["as_of_date"],
        "status": "candidate",
        "source_task_key": task["task_id"],
        "nuance_tag": task["nuance_tag"],
        "persona_used": task.get("persona_hint"),
    }


def verify_nuance_in_batches(candidates, corpus, config):
    verified = []
    rejected = Counter()
    batch_size = int(config.get("nuance_verify_batch_size", 6) or 6)
    for start in range(0, len(candidates), batch_size):
        batch = [tag_minimal_evidence(qa, corpus) for qa in candidates[start:start + batch_size]]
        valid_batch, batch_rejected = filter_valid_answerable_qas(batch, corpus)
        rejected.update(batch_rejected)
        if not valid_batch:
            continue
        checked = verify_qas(client, valid_batch, corpus, config)
        checked, checked_rejected = filter_valid_answerable_qas(checked, corpus)
        rejected.update(checked_rejected)
        verified.extend(checked)
    return verified, rejected

nuance_tasks = [task for task in build_nuance_tasks(corpus, CONFIG) if task["task_id"] not in done_task_keys]
nuance_unit_ids = {uid for task in nuance_tasks for uid in task.get("evidence_unit_ids", [])}
hydrated_nuance_chunks = hydrate_chunks_for_units(corpus, nuance_unit_ids)
nuance_tasks = [task for task in nuance_tasks
                if all(unit_has_valid_chunk(corpus, uid, CONFIG.get("min_chunk_chars_for_seed", 80))
                       for uid in task.get("evidence_unit_ids", []))]
print(f"Nuance tasks: {len(nuance_tasks)}; hydrated nuance chunks: {hydrated_nuance_chunks}; expected calls ~{len(nuance_tasks) + math.ceil(len(nuance_tasks) / CONFIG.get('nuance_verify_batch_size', 6))}")
print("Nuance task distribution:", dict(Counter(task["nuance_tag"] for task in nuance_tasks)))

nuance_candidates = []
with ThreadPoolExecutor(max_workers=CONFIG.get("max_workers", 1)) as executor:
    futures = {executor.submit(generate_nuance_qa, client, task, corpus, CONFIG): task for task in nuance_tasks}
    for future in tqdm(as_completed(futures), total=len(futures)):
        try:
            qa = future.result()
        except Exception as e:
            print(f"Nuance generate failed: {e}")
            continue
        if qa and qa.get("question"):
            nuance_candidates.append(qa)

nuance_verified, nuance_gt_rejections = verify_nuance_in_batches(nuance_candidates, corpus, CONFIG)
for qa in nuance_verified:
    if qa.get("qa_id") not in done_ids:
        append_checkpoint(CONFIG["checkpoint_path"], qa)

nuance_generation_stats = {
    "tasks_built": len(nuance_tasks),
    "generated": len(nuance_candidates),
    "verified_records_appended": len(nuance_verified),
    "task_distribution": dict(Counter(task["nuance_tag"] for task in nuance_tasks)),
    "generated_by_category": dict(Counter(qa.get("category") for qa in nuance_candidates)),
    "verified_by_category": dict(Counter(qa.get("category") for qa in nuance_verified)),
    "verified_by_nuance_tag": dict(Counter(qa.get("nuance_tag") for qa in nuance_verified)),
    "ground_truth_validation_rejections": dict(nuance_gt_rejections),
    "api_requests_used_after_nuance": RATE_LIMIT_STATE["total_requests"],
}
print(f"Nuance generated: {len(nuance_candidates)}; verified records appended: {len(nuance_verified)}")
print("Nuance generated by category:", nuance_generation_stats["generated_by_category"])
print("Nuance ground-truth rejections:", nuance_generation_stats["ground_truth_validation_rejections"])

## 14. Loc: chi giu QA pass verification + Diversity control

In [ ]:
all_candidates = []
checkpoint_path = Path(CONFIG["checkpoint_path"])
checkpoint_invalid_counts = Counter()
if checkpoint_path.exists():
    with open(checkpoint_path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                qa = json.loads(line)
                ok, reason = validate_ground_truth_for_save(qa, corpus)
                if ok:
                    all_candidates.append(qa)
                else:
                    checkpoint_invalid_counts[reason] += 1

# qa_candidates = all generated+verified attempts, including rejected. This preserves the old
# checkpoint semantics while making the intermediate artifact explicit.
with open(CONFIG["qa_candidates_path"], "w", encoding="utf-8") as f:
    for qa in all_candidates:
        f.write(json.dumps(qa, ensure_ascii=False) + "\n")

passed_qa = [qa for qa in all_candidates if qa.get("verifier_result") == "pass"]
with open(CONFIG["qa_verified_path"], "w", encoding="utf-8") as f:
    for qa in passed_qa:
        f.write(json.dumps(qa, ensure_ascii=False) + "\n")

verifier_stats = {
    "checkpoint_records_valid_for_filtering": len(all_candidates),
    "checkpoint_invalid_ground_truth": dict(checkpoint_invalid_counts),
    "passed_verification": len(passed_qa),
    "verifier_pass_rate": len(passed_qa) / max(1, len(all_candidates)),
    "passed_by_category": dict(Counter(qa.get("category") for qa in passed_qa)),
}
print(f"Pass verification: {len(passed_qa)} / {len(all_candidates)} ({verifier_stats['verifier_pass_rate']:.2%})")
print(f"Saved qa_candidates: {CONFIG['qa_candidates_path']}")
print(f"Saved qa_verified: {CONFIG['qa_verified_path']}")
if checkpoint_invalid_counts:
    print("Checkpoint records skipped due to invalid ground truth:", dict(checkpoint_invalid_counts))

def enforce_diversity(qa_list, target_total, config):
    # Safety net only. The main diversity mechanism is bundle-level generation planning.
    per_provision = Counter()
    per_document = Counter()
    removal_counts = Counter()
    max_per_doc = max(1, int(target_total * config["max_qa_per_document_ratio"]))
    max_per_provision = int(config.get("max_qa_per_provision", 2) or 2)
    kept = []
    for qa in qa_list:
        ground_truth = qa.get("ground_truth", {})
        provision_ids = ground_truth.get("provision_ids", [])
        doc_keys = ground_truth.get("document_ids", [])
        uid_key = tuple(sorted(provision_ids)) or (qa.get("qa_id"),)
        if provision_ids and any(per_provision[uid] >= max_per_provision for uid in provision_ids):
            removal_counts["max_qa_per_provision"] += 1
            continue
        if any(per_document[d] >= max_per_doc for d in doc_keys):
            removal_counts["max_qa_per_document"] += 1
            continue
        if len(kept) >= target_total:
            removal_counts["target_total_cap"] += 1
            continue
        kept.append(qa)
        for uid in provision_ids:
            per_provision[uid] += 1
        if not provision_ids:
            per_provision[uid_key] += 1
        for d in doc_keys:
            per_document[d] += 1
    stats = {
        "input": len(qa_list),
        "kept": len(kept),
        "removed_total": sum(removal_counts.values()),
        "removed_by_rule": dict(removal_counts),
        "max_qa_per_provision": max_per_provision,
        "max_qa_per_document": max_per_doc,
    }
    return kept, stats

diverse_qa, diversity_filter_stats = enforce_diversity(passed_qa, CONFIG["target_total"], CONFIG)
print(f"Sau safety-net diversity control: {len(diverse_qa)}")
print("Removed by diversity rule:", diversity_filter_stats["removed_by_rule"])
print(Counter(qa.get("category") for qa in diverse_qa))

## 15. Dedup (nhac lai: chay dedup 2 lop bang embedding o buoc rieng, ngoai notebook nay,
vi can them sentence-transformers -- xem `dedup_filter.py` trong installation guide da gui truoc do).
O day chi loc trung tuyet doi theo cau hoi giong het nhau.

In [ ]:
seen_questions = set()
final_qa = []
dedup_removed = 0
for qa in diverse_qa:
    key = qa.get("question", "").strip().lower()
    if CONFIG.get("dedup_exact_questions", True) and key in seen_questions:
        dedup_removed += 1
        continue
    seen_questions.add(key)
    final_qa.append(qa)

dedup_stats = {
    "input": len(diverse_qa),
    "kept": len(final_qa),
    "removed_exact_duplicate_question": dedup_removed,
}
print(f"Sau loc trung tuyet doi: {len(final_qa)}")
print("Dedup removed:", dedup_removed)

## 16. Export ket qua

In [ ]:
INTERNAL_QA_FIELDS = {"source_task_key"}

def export_qa_record(qa):
    return {k: v for k, v in qa.items() if k not in INTERNAL_QA_FIELDS}

with open(CONFIG["qa_final_path"], "w", encoding="utf-8") as f:
    for qa in final_qa:
        f.write(json.dumps(export_qa_record(qa), ensure_ascii=False) + "\n")

# Backward-compatible filename used by older notebook runs/downloads.
out_path = Path(CONFIG["output_dir"]) / "qa_candidates_verified.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for qa in final_qa:
        f.write(json.dumps(export_qa_record(qa), ensure_ascii=False) + "\n")

final_distribution = {
    "category_distribution": dict(Counter(qa.get("category") for qa in final_qa)),
    "difficulty_distribution": dict(Counter(qa.get("difficulty") for qa in final_qa)),
    "nuance_distribution": dict(Counter(qa.get("nuance_tag") for qa in final_qa if qa.get("nuance_tag"))),
    "main_qa_count": sum(1 for qa in final_qa if not qa.get("nuance_tag")),
    "nuance_qa_count": sum(1 for qa in final_qa if qa.get("nuance_tag")),
}

run_report = {
    "sampling": globals().get("sampling_stats", {}),
    "bundle_generation": globals().get("bundle_generation_stats", {}),
    "main_generation": globals().get("main_generation_stats", {}),
    "nuance_generation": globals().get("nuance_generation_stats", {}),
    "verifier": globals().get("verifier_stats", {}),
    "diversity_filter": globals().get("diversity_filter_stats", {}),
    "dedup": globals().get("dedup_stats", {}),
    "final": {
        "final_qa_count": len(final_qa),
        **final_distribution,
    },
    "config": {
        "main_target_total": CONFIG.get("main_target_total"),
        "nuance_target_total": CONFIG.get("nuance_target_total"),
        "target_total": CONFIG.get("target_total"),
        "sampled_provisions_target": CONFIG.get("sampled_provisions_target"),
        "min_provisions_per_sampled_doc": CONFIG.get("min_provisions_per_sampled_doc"),
        "max_provisions_per_sampled_doc": CONFIG.get("max_provisions_per_sampled_doc"),
        "context_units_per_task": CONFIG.get("context_units_per_task"),
        "batch_size": CONFIG.get("batch_size"),
        "oversample_factor": CONFIG.get("oversample_factor"),
        "max_qa_per_provision": CONFIG.get("max_qa_per_provision"),
        "max_qa_per_document_ratio": CONFIG.get("max_qa_per_document_ratio"),
        "nuance_quota": CONFIG.get("nuance_quota"),
    },
    "outputs": {
        "qa_candidates_path": CONFIG.get("qa_candidates_path"),
        "qa_verified_path": CONFIG.get("qa_verified_path"),
        "qa_final_path": CONFIG.get("qa_final_path"),
        "compatible_final_path": str(out_path),
        "run_report_path": CONFIG.get("run_report_path"),
        "run_report_md_path": CONFIG.get("run_report_md_path"),
    },
}

with open(CONFIG["run_report_path"], "w", encoding="utf-8") as f:
    json.dump(run_report, f, ensure_ascii=False, indent=2)

report_lines = [
    "# QA Generation Run Report",
    "",
    "## Sampling",
    f"- Sampled documents: {run_report['sampling'].get('sampled_documents', 0)} / {run_report['sampling'].get('total_documents', 0)}",
    f"- Sampled provisions with valid chunks: {run_report['sampling'].get('sampled_provisions_with_valid_chunks', 0)} / {run_report['sampling'].get('total_provisions', 0)}",
    f"- Average provisions/sample document: {run_report['sampling'].get('average_provisions_per_sampled_document', 0):.2f}",
    "",
    "## Generation",
    f"- Main raw generated by category: {run_report['main_generation'].get('raw_generated_by_category', {})}",
    f"- Nuance generated by category: {run_report['nuance_generation'].get('generated_by_category', {})}",
    "",
    "## Verification",
    f"- Pass rate: {run_report['verifier'].get('verifier_pass_rate', 0):.2%}",
    f"- Passed by category: {run_report['verifier'].get('passed_by_category', {})}",
    "",
    "## Filtering",
    f"- Removed by diversity rule: {run_report['diversity_filter'].get('removed_by_rule', {})}",
    f"- Removed exact duplicate questions: {run_report['dedup'].get('removed_exact_duplicate_question', 0)}",
    "",
    "## Final",
    f"- Final QA count: {len(final_qa)}",
    f"- Category distribution: {final_distribution['category_distribution']}",
    f"- Nuance distribution: {final_distribution['nuance_distribution']}",
]
with open(CONFIG["run_report_md_path"], "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines) + "\n")

meta = {
    "num_qa": len(final_qa),
    **final_distribution,
    "main_target_total": CONFIG.get("main_target_total"),
    "nuance_target_total": CONFIG.get("nuance_target_total"),
    "corpus_version": CONFIG["corpus_version"],
    "generator_model": CONFIG["generator_model"],
    "verifier_model": CONFIG["verifier_model"],
    "generation_strategy": "document_first_related_context_bundle_multi_category_with_integrated_nuance",
    "batch_size": CONFIG.get("batch_size"),
    "context_units_per_task": CONFIG.get("context_units_per_task"),
    "oversample_factor": CONFIG.get("oversample_factor"),
    "requests_per_minute": CONFIG.get("requests_per_minute"),
    "requests_per_day": CONFIG.get("requests_per_day"),
    "max_api_requests": CONFIG.get("max_api_requests"),
    "nuance_quota": CONFIG.get("nuance_quota"),
    "nuance_oversample_factor": CONFIG.get("nuance_oversample_factor"),
    "nuance_verify_batch_size": CONFIG.get("nuance_verify_batch_size"),
    "diversity_filter": globals().get("diversity_filter_stats", {}),
    "dedup": globals().get("dedup_stats", {}),
    "qa_candidates_path": CONFIG.get("qa_candidates_path"),
    "qa_verified_path": CONFIG.get("qa_verified_path"),
    "qa_final_path": CONFIG.get("qa_final_path"),
    "run_report_path": CONFIG.get("run_report_path"),
    "run_report_md_path": CONFIG.get("run_report_md_path"),
}
with open(Path(CONFIG["output_dir"]) / "qa_candidates_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(json.dumps(meta, indent=2, ensure_ascii=False))
print(f"\nDa luu final: {CONFIG['qa_final_path']}")
print(f"Da luu compatible export: {out_path}")
print(f"Da luu run report: {CONFIG['run_report_path']}")
print(f"Da luu markdown report: {CONFIG['run_report_md_path']}")
print("Buoc tiep theo (ngoai notebook nay): dedup 2 lop bang embedding, Retrieval Sanity Check tren mau 10-15%, Human Spot Check.")
